In [1]:
import pyrforest
pretty_print_default()

In [2]:
load('trace_af.sage')

40
{13: [2174 1542 1649  787    7  132 1771 1683]
[ 583 1652 1842 1181 1920  147  560 1818]
[2122 2140  518 2178 1008 1159  653 1956]
[ 240 1123 1048 2010  214  200  217 1158]
[ 680 2070  948 1003   79    1   77  191]
[1362 1772  789 1751  673 1590 2056 1462]
[1871 1065  574 1193  849 1909 2101  914]
[1004 1268  798 1817 1259 1607 1789 1799], 17: [1448 3217 2353  931 2236 1368 2739   94]
[3605 1426 2440  168 3746 1027 2422 2346]
[3003 3661 2695 2418  340 1761 3236 3483]
[ 760 3196  213  314 1254 1372  917 4853]
[1720 3334 3479 1410 1704  643 2196 3771]
[1407 4825 4440 3569 3068 4901 2668 4745]
[3403 4436 1802 1644   22 3933 2733   47]
[2109 2361 4565  440 4205  364  450 4199], 19: [3574 2958 6274 4503 5355 3861  915 4591]
[6214  500 4953 5068  374 3161 2970 2007]
[ 218 3964 2238 2594 4394 1566 5828 6635]
[4766 4607 2901 5035 1534 3604 4869  419]
[3109 1537 3152 2753 2872 6011 2486 6067]
[1908 6595 3491 1039 3262  174 4319 3726]
[6373 6635 4948 5904 2330 4524 5555  130]
[1842 4700 5952 

In [3]:
def random_polynomial(n):
    R.<x> = ZZ[]
    while True:
        f = R.random_element(degree=n)
        if f[0] != 0:
            return f

In [4]:
def A_F_l_lambda(C, lam, l):
    K = C.base_ring()
    p = K.characteristic()
    assert K.degree() == 1
    g = C.genus()

    f, _ = C.hyperelliptic_polynomials()
    F = f.change_ring(Zmod(p^lam))
    H = F^((p - 1) // 2)

    H_l = H^(2 * l + 1)
    size = (2 * l + 1)  * (g + 1) + 1
    rows = [[H_l[v * p - u] for u in range(size)] for v in range(size)]
    M = Matrix(rows)
    assert M.dimensions() == (size, size)
    return M

In [5]:
@cached_method
def phi_lambda(lam):
    assert lam >= 1
    R.<t> = QQ[]
    phi = R(0)
    for j in range(lam):
        term = 1 / (2^(lam + j))
        term *= binomial(lam + j - 1, j)
        term *= (1 - t)^j * (1 + t)^lam - (1 + t)^j * (1 - t)^lam
        phi += term
    return phi

In [6]:
def trace_formula(C, r, lam):
    p = C.base_ring().characteristic()
    assert lam >= 1
    assert p >= 2 * lam + 1
    R = Zmod(p^lam)
    total = R(1)
    phi = phi_lambda(lam)
    A_tilde = list(compute_A_tilde_F(C, lam))
    for l in range(lam):
        phi_coeff = phi[2 * l + 1]
        phi_coeff = R(phi_coeff.numerator()) / R(phi_coeff.denominator())
        A_tilde_l = A_tilde[l]
        total += phi_coeff * (A_tilde_l^r).trace()
    return total * (p^r - 1)

In [7]:
def count_points(C, r, lam):
    compute
    
    count = trace_formula(C, r, lam)
    p = C.base_ring().characteristic()
    Cpr = C.change_ring(GF(p, r))
    f, _ = Cpr.hyperelliptic_polynomials()
    g = Cpr.genus()

    if f[0] == 0:
        count += 1
    elif f[0].is_square():
        count += 2
        
    if f[2 * g + 2] == 0:
        count += 1
    elif f[2 * g + 2].is_square():
        count += 2
    return count

In [61]:
def count_points(F, N, r, lam):
    assert lam >= 1
    assert r >= 1

    phi = phi_lambda(lam)
    prime_point_counts = dict()
    for ell in range(lam):
        for p, A_F_l in compute_A_F_l_avg_poly(F, N, lam, ell).items():
        #for p in primes(30, N):
            fpr = F.change_ring(GF(p, r))
            if not fpr.is_squarefree():
                continue
            
            assert p >= 2 * lam + 1
            R = Zmod(p^lam)
            if p not in prime_point_counts:
                prime_point_counts[p] = R(1)
            Fp = F.change_ring(R)
            H_0 = Fp[0] ** 2
            H_max = Fp.leading_coefficient() ** 2

            # Smart computation
            trace_A_tilde_F_l_r = Integer(H_0 ** ((2 * ell + 1) * r) + H_max ** ((2 * ell + 1) * r) + (A_F_l**r).trace())

            # Naive computation
            Afllam = A_F_l_lambda(HyperellipticCurve(F.change_ring(Zmod(p))), lam, ell)
            tr = (Afllam^r).trace()
            
            phi_coeff = phi[2 * ell + 1]
            phi_coeff = Integer(R(phi_coeff.numerator()) / R(phi_coeff.denominator()))
            #prime_point_counts[p] += (phi_coeff * trace_A_tilde_F_l_r) % p^lam
            prime_point_counts[p] += phi_coeff * tr

    for p in prime_point_counts:
        prime_point_counts[p] *= p^r - 1

        Kpr = GF(p, r)
        fpr = F.change_ring(Kpr)
        Cpr = HyperellipticCurve(fpr)
        g = Cpr.genus()
    
        if fpr[0] == 0:
            prime_point_counts[p] += 1
        elif fpr[0].is_square():
            prime_point_counts[p] += 2
            
        if fpr[2 * g + 2] == 0:
            prime_point_counts[p] += 1
        elif fpr[2 * g + 2].is_square():
            prime_point_counts[p] += 2
    return prime_point_counts

f = random_polynomial(4)
count_points(f, 40, 2, 3)

{13: 192, 17: 288, 19: 384, 23: 576, 29: 896, 31: 1008, 37: 1440}

In [65]:
lam = 2
r = 2
N = 37  # Noam prime
f = random_polynomial(5)
print('Counting')
counts = count_points(f, N, r, lam)
for p, c in counts.items():
    assert p.is_prime()
    K = GF(p)
    fp = f.change_ring(K)
    C = HyperellipticCurve(fp)
    assert (C.count_points(r)[-1] % p^lam == c)
print('Verified')

Counting
Verified
